# Air Quality in Dar es Salaam 🇹🇿

In [ ]:
# Import libraries here
import inspect
import time
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns
from pymongo import MongoClient
from sklearn.metrics import mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.ar_model import AutoReg

warnings.filterwarnings("ignore")

## Prepare Data

In [ ]:
## Connect to MongoDB server running at host "localhost" on port 27017. Then connect to the "air-quality" database and assign the collection 
## for Dar es Salaam to the variable name dar.

client = MongoClient(host ="localhost", port=27017)
db = client["air-quality"]
dar = db["dar-es-salaam"]

### Explore

In [ ]:
## Determine the numbers assigned to all the sensor sites in the Dar es Salaam collection. Your submission should be a list of integers. 

sites = dar.distinct("metadata.site")
sites

In [ ]:
## Determine which site in the Dar es Salaam collection has the most sensor readings (of any type, not just PM2.5 readings). 
## You submission readings_per_site should be a list of dictionaries that follows this format:
## [{'_id': 6, 'count': 70360}, {'_id': 29, 'count': 131852}]

result = dar.aggregate(
    [
        {"$group": {"_id": "$metadata.site", "count": {"$count": {}}}}
    ]
)
readings_per_site = list(result)
readings_per_site

### Import

**Create** a wrangle function that will extract the PM2.5 readings from the site that has the most total readings in the Dar es Salaam collection. 
Your function should do the following steps:

##### Localize reading time stamps to the timezone for "Africa/Dar_es_Salaam".
##### Remove all outlier PM2.5 readings that are above 100.
##### Resample the data to provide the mean PM2.5 reading for each hour.
##### Impute any missing values using the forward-fill method.
##### Return a Series y.

In [ ]:
def wrangle(collection):
    # Select data for the site with the most readings (site 11)
    results = collection.find(
        {"metadata.site": 11, "metadata.measurement": "P2"},
        projection={"P2": 1, "timestamp": 1, "_id": 0},
    )

    # Convert to DataFrame and set timestamp as index
    df = pd.DataFrame(list(results)).set_index("timestamp")

    # Convert index to datetime if needed
    df.index = pd.to_datetime(df.index)

    # Localize and convert timezone
    df.index = df.index.tz_localize("UTC").tz_convert("Africa/Dar_es_Salaam")

    # Remove outliers (P2 > 100)
    df = df[df["P2"] <= 100]

    # Resample and forward-fill missing values
    y = df["P2"].resample("1H").mean().ffill()

    return y


In [ ]:
## Use your wrangle function to query the dar collection and return your cleaned results.

y = wrangle(dar)
y.head()

In [ ]:
## Create a time series plot of the readings in y. Label your x-axis "Date" and your y-axis "PM2.5 Level". Use the title "Dar es Salaam PM2.5 Levels".

fig, ax = plt.subplots(figsize=(15, 6))
y.plot(xlabel= "Date", ylabel= "PM2.5 Level", title="Dar es Salaam PM2.5 Levels", ax =ax);

In [ ]:
## Plot the rolling average of the readings in y. Use a window size of 168 (the number of hours in a week). 
## Label your x-axis "Date" and your y-axis "PM2.5 Level". Use the title "Dar es Salaam PM2.5 Levels, 7-Day Rolling Average".

fig, ax = plt.subplots(figsize=(15, 6))
y.rolling(168).mean().plot(ax=ax, xlabel="Date", ylabel="PM2.5 Level", title="Dar es Salaam PM2.5 Levels, 7-Day Rolling Average");
​

In [ ]:
##Create an ACF plot for the data in y. Be sure to label the x-axis as "Lag [hours]" and the y-axis as "Correlation Coefficient". 
## Use the title "Dar es Salaam PM2.5 Readings, ACF".

fig, ax = plt.subplots(figsize=(15, 6))
plot_acf(y, ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient")
plt.title("Dar es Salaam PM2.5 Readings, ACF");

In [ ]:
##Create an ACF plot for the data in y. Be sure to label the x-axis as "Lag [hours]" and the y-axis as "Correlation Coefficient". 
## Use the title "Dar es Salaam PM2.5 Readings, ACF".

fig, ax = plt.subplots(figsize=(15, 6))
plot_pacf(y, ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient")
plt.title("Dar es Salaam PM2.5 Readings, PACF");




### Split

In [ ]:
## Split y into training and test sets. The first 90% of the data should be in your training set. The remaining 10% should be in the test set.

cutoff_test = int(len(y) * 0.90)
y_train = y.iloc[:cutoff_test]
y_test = y.iloc[cutoff_test:]
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Build Model

##### Baseline

In [ ]:
## Establish the baseline mean absolute error for your model.

y_train_mean = y_train.mean()
y_pred_baseline = [y_train_mean] * len(y_train)
mae_baseline = mean_absolute_error(y_train, y_pred_baseline)
print("Mean P2 Reading:", y_train_mean)
print("Baseline MAE:", mae_baseline)

### Iterate

In [ ]:
## You're going to use an AutoReg model to predict PM2.5 readings, but which hyperparameter settings will give you the best performance? 
## Use a for loop to train your AR model on using settings for lags from 1 to 30. Each time you train a new model, 
## calculate its mean absolute error and append the result to the list maes. Then store your results in the Series mae_series.

# Create range to test different lags
p_params = range(1, 31)

# Create empty list to hold mean absolute error scores
maes = []

# Iterate through all values of p in `p_params`
for p in p_params:
    # Build model
    model = AutoReg(y_train, lags=p).fit()

    # Make predictions on training data, dropping null values caused by lag
    y_pred = model.predict().dropna()

    # Calculate mean absolute error for training data vs predictions
    mae = mean_absolute_error(y_train.iloc[p:], y_pred)

    # Append `mae` to list `maes`
    maes.append(mae)

# Put list `maes` into Series with index `p_params`
mae_series = pd.Series(maes, name="mae", index=p_params)

# Inspect head of Series
mae_series.head()

In [ ]:
## Look through the results in mae_series and determine what value for p provides the best performance. Then build and train best_model 
## using the best hyperparameter value.

## Note: Make sure that you build and train your model in one line of code, and that the data type of best_model is 
## statsmodels.tsa.ar_model.AutoRegResultsWrapper.

from statsmodels.tsa.ar_model import AutoReg

# Get the best lag value (smallest MAE)
best_p = mae_series.idxmin()

# Build and train the best model in one line
best_model = AutoReg(y_train, lags=best_p).fit()

In [ ]:
## Calculate the training residuals for best_model and assign the result to y_train_resid. Note that your name of your Series should be "residuals".

y_train_resid = best_model.resid
y_train_resid.name = "residuals"
y_train_resid.head()

In [ ]:
## Create a histogram of y_train_resid. Be sure to label the x-axis as "Residuals" and the y-axis as "Frequency". Use the title "Best Model, 
## Training Residuals".

# Plot histogram of residuals
fig, ax = plt.subplots()

# Pass ax=ax so the plot uses the correct axes
y_train_resid.hist(ax=ax)

ax.set_xlabel("Residuals")
ax.set_ylabel("Frequency")
ax.set_title("Best Model, Training Residuals")


In [ ]:
## Create an ACF plot for y_train_resid. Be sure to label the x-axis as "Lag [hours]" and y-axis as "Correlation Coefficient". 
### Use the title "Dar es Salaam, Training Residuals ACF".

fig, ax = plt.subplots(figsize=(15, 6))
plot_acf(y_train_resid, ax=ax)
plt.xlabel("Lag [hours]")
plt.ylabel("Correlation Coefficient")
plt.title("Dar es Salaam, Training Residuals ACF");

### Evaluate

In [ ]:
## Perform walk-forward validation for your model for the entire test set y_test. Store your model's predictions in the Series y_pred_wfv. 
## Make sure the name of your Series is "prediction" and the name of your Series index is "timestamp".

y_pred_wfv = []
​
# Copy the training data into history (this will grow over time)
history = y_train.copy()
​
# Perform walk-forward validation
for i in range(len(y_test)):
    # Fit model on the current history
    model = AutoReg(history, lags=best_p).fit()
    
    # Forecast the next time step (1 step ahead)
    y_hat = model.predict(start=len(history), end=len(history))
    
    # Store prediction
    y_pred_wfv.append(y_hat.iloc[0])
    
    # Update history with the true observation from the test set
    history = pd.concat([history, pd.Series([y_test.iloc[i]], index=[y_test.index[i]])])
​
# Convert predictions to Series with same index as y_test
y_pred_wfv = pd.Series(y_pred_wfv, index=y_test.index, name="prediction")
y_pred_wfv.index.name = "timestamp"
​
# Preview
y_pred_wfv.head()

In [ ]:
# Enter y_pred_wfv at ... (Ellipsis) to see the test mean absolute error

test_mae = mean_absolute_error(y_test, y_pred_wfv)

print("Test MAE (walk forward validation):", round(test_mae, 2))

### Communicate Results

In [ ]:
# Task 3.5.18: Put the values for y_test and y_pred_wfv into the DataFrame df_pred_test (don't forget the index). Then plot df_pred_test 
# using plotly express. In the legend, your lines should be labeled "y_test" and "y_pred_wfv". Be sure to label the x-axis as "Date" 
# and the y-axis as "PM2.5 Level". Use the title "Dar es Salaam, WFV Predictions".

df_pred_test = ...
fig = ...
fig.update_layout(
    title="Dar es Salaam, WFV Predictions",
    xaxis_title="Date",
    yaxis_title="PM2.5 Level",
)
# Don't delete the code below 👇
fig.write_image("images/3-5-18.png", scale=1, height=500, width=700)
​
fig.show()